# E0. Verificación del pipeline de YOLO26 en Kaggle

Corrida corta (10 épocas) con YOLO26n para comprobar que todo funciona de punta a punta: lectura del dataset, entrenamiento, validación global y por fuente, guardado de pesos. Las métricas de esta corrida **no cuentan**; lo que interesa es el tiempo por época y que no haya avisos de imágenes descartadas.

Requisitos en la configuración del cuaderno (panel derecho): acelerador GPU e **Internet activado** (hace falta para `pip` y para descargar `yolo26n.pt`). Dataset adjunto: `deteccionoso`.

Diseño de pruebas: `docs/experimentos.md`, parte 1.

## 1. Entorno

In [ ]:
%pip install -q ultralytics==8.4.138

import sys, platform, torch, ultralytics
print("Python     :", sys.version.split()[0])
print("torch      :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0), "| n GPUs:", torch.cuda.device_count())
print("Ultralytics:", ultralytics.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Dataset

Conteos esperados: train 2656/2656, val 598/598, test 578/578. Listas: val_inat 260, val_ena24 338, test_inat 260, test_ena24 318.

In [ ]:
from pathlib import Path

DATASET = Path("/kaggle/input/deteccionoso")
print("Raíz:", sorted(p.name for p in DATASET.iterdir()))
for conj in ("train", "val", "test"):
    n_img = sum(1 for _ in (DATASET / "images" / conj).glob("*.jpg"))
    n_lbl = sum(1 for _ in (DATASET / "labels" / conj).glob("*.txt"))
    print(f"{conj:<6} {n_img} imágenes, {n_lbl} etiquetas")
for lista in ("val_inat.txt", "val_ena24.txt", "test_inat.txt", "test_ena24.txt"):
    n = sum(1 for _ in open(DATASET / lista, encoding="utf-8"))
    print(f"{lista:<16} {n} rutas")

## 3. Archivos yaml con la ruta de Kaggle

Los yaml del dataset traen la ruta absoluta del equipo local en `path`. Aquí se escriben de nuevo en `/kaggle/working/cfg/` apuntando a `/kaggle/input/deteccionoso`. Las listas de rutas no cambian: son relativas a la raíz del dataset.

In [ ]:
CFG = Path("/kaggle/working/cfg")
CFG.mkdir(parents=True, exist_ok=True)

def escribir_yaml(nombre, val, test):
    ruta = CFG / nombre
    ruta.write_text(
        f"path: {DATASET}\n"
        f"train: images/train\n"
        f"val: {val}\n"
        f"test: {test}\n"
        f"nc: 1\n"
        f"names: ['oso']\n",
        encoding="utf-8",
    )
    return str(ruta)

YAML = {
    "global":     escribir_yaml("dataset.yaml",            "images/val",    "images/test"),
    "val_inat":   escribir_yaml("dataset_val_inat.yaml",   "val_inat.txt",  "val_inat.txt"),
    "val_ena24":  escribir_yaml("dataset_val_ena24.yaml",  "val_ena24.txt", "val_ena24.txt"),
    "test_inat":  escribir_yaml("dataset_test_inat.yaml",  "test_inat.txt", "test_inat.txt"),
    "test_ena24": escribir_yaml("dataset_test_ena24.yaml", "test_ena24.txt","test_ena24.txt"),
}
print(open(YAML["global"]).read())

## 4. Entrenamiento corto (E0)

Argumentos del protocolo (`docs/experimentos.md`, 1.3) salvo `epochs=10`. `cache="ram"` guarda las imágenes ya reescaladas en memoria para no decodificar los JPEG de 3 MP de ENA24 en cada época; no cambia el resultado, solo el tiempo. `/kaggle/input` es de solo lectura, así que es normal ver un aviso de que no puede escribir la caché de etiquetas (`labels/train.cache`); no afecta.

Qué mirar en la salida:
- Las líneas `train: Scanning ...` y `val: Scanning ...`: deben decir 2656 y 598 imágenes y **0 corrupt**. Si aparece algún `ignoring corrupt image/label`, anotar cuál.
- El optimizador que elige `optimizer=auto` (lo imprime al inicio).
- El tiempo por época (para planificar E1 a E4).

In [ ]:
import time
from ultralytics import YOLO

RUNS = "/kaggle/working/runs"
t0 = time.time()
model = YOLO("yolo26n.pt")
model.train(
    data=YAML["global"],
    epochs=10,
    imgsz=640,
    batch=16,
    seed=0,
    deterministic=True,
    cache="ram",
    device=0,
    project=RUNS,
    name="e0_yolo26n_verif",
    exist_ok=True,
)
print(f"\nTiempo total de entrenamiento: {(time.time() - t0) / 60:.1f} min")

## 5. Validación en tres particiones con `best.pt`

Aquí se prueba el mecanismo de D15 en Kaggle: la misma validación sobre validación global, solo ENA24 y solo iNaturalist. Los números son de un modelo de 10 épocas y no significan nada todavía.

In [ ]:
import pandas as pd

BEST = f"{RUNS}/e0_yolo26n_verif/weights/best.pt"
m = YOLO(BEST)
filas = []
for nombre in ("global", "val_ena24", "val_inat"):
    r = m.val(data=YAML[nombre], imgsz=640, batch=16, device=0, plots=False,
              project=RUNS, name=f"e0_val_{nombre}", exist_ok=True)
    d = r.results_dict
    filas.append({
        "partición": nombre,
        "P": d["metrics/precision(B)"],
        "R": d["metrics/recall(B)"],
        "mAP50": d["metrics/mAP50(B)"],
        "mAP50-95": d["metrics/mAP50-95(B)"],
    })
tabla = pd.DataFrame(filas).round(3)
print(tabla.to_string(index=False))
tabla.to_csv("/kaggle/working/e0_val_por_particion.csv", index=False)

## 6. Resumen para `docs/experimentos.md`

Copiar la salida de esta celda en la entrada E0. Los pesos y `results.csv` quedan en `/kaggle/working/` y se descargan desde el panel *Output* del cuaderno.

In [ ]:
import shutil, glob
from pathlib import Path

shutil.copy(BEST, "/kaggle/working/e0_yolo26n_best.pt")
shutil.copy(f"{RUNS}/e0_yolo26n_verif/results.csv", "/kaggle/working/e0_results.csv")

res = pd.read_csv("/kaggle/working/e0_results.csv")
res.columns = [c.strip() for c in res.columns]
print("Épocas registradas:", len(res))
print("Columnas:", list(res.columns))
print()
print(res.tail(3).round(4).to_string(index=False))
print()
print("Archivos en /kaggle/working:")
for p in sorted(Path("/kaggle/working").glob("*")):
    print("  ", p.name)